In [1]:
!pip install transformers torch

   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   -------------- ------------------------- 4.2/11.6 MB 21.1 MB/s eta 0:00:01
   ------------------------------------- -- 10.7/11.6 MB 26.2 MB/s eta 0:00:01
   ---------------------------------------- 11.6/11.6 MB 24.4 MB/s  0:00:00
   ---------------------------------------- 0.0/561.5 kB ? eta -:--:--
   ---------------------------------------- 561.5/561.5 kB 36.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 24.5 MB/s  0:00:00

   ---------------------------------------- 0/5 [safetensors]
   -------- ------------------------------- 1/5 [regex]
   ---------------- ----------------------- 2/5 [huggingface-hub]
   ---------------- ----------------------- 2/5 [huggingface-hub]
   ---------------- ----------------------- 2/5 [huggingface-hub]
   ---------------- ----------------------- 2/5 [huggingface-hub]
   ---------------- --

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. 모델과 토크나이저 불러오기
model_name = "skt/kogpt2-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPU 사용 가능 시 이동
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 2. 대화 함수 정의
def chat():
    print("=== 한국어 GPT-2 대화형 챗봇 (개선 버전) ===")
    print("종료하려면 'quit' 입력\n")

    chat_history_ids = None # 토큰화된 대화 기록 저장용
    max_history_tokens = 512 # 최대 대화 기록 토큰 수

    while True:
        user_input = input("User: ")
        if user_input.lower() == "quit":
            print("Bot: 대화를 종료합니다. 안녕히 가세요!")
            break

        # 사용자의 입력 토큰화
        new_input_ids = tokenizer(f"User: {user_input}\nBot:", return_tensors="pt").input_ids.to(device)

        # 기존 대화 기록이 있으면 합치기
        if chat_history_ids is None:
            chat_history_ids = new_input_ids
        else:
            # 대화 기록 길이 제한 (최대 512 토큰)
            # 새로운 입력을 추가했을 때 512를 넘지 않도록, 오래된 기록을 자름
            total_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
            if total_ids.shape[-1] > max_history_tokens:
                chat_history_ids = total_ids[:, -max_history_tokens:] # 최근 512개 토큰만 남김
            else:
                chat_history_ids = total_ids

        # 답변 생성
        outputs = model.generate(
            input_ids=chat_history_ids,
            max_length=len(chat_history_ids[0]) + 100, # 답변 길이를 동적으로 설정
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            repetition_penalty=1.2
        )

        # 결과 디코딩 및 추출
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        bot_reply = response.split("Bot:")[-1].split("User:")[0].strip()

        print(f"Bot: {bot_reply}")

        # 대화 기록 업데이트 (다음 턴을 위해)
        # 생성된 답변까지 포함하여 다음 입력으로 사용
        chat_history_ids = tokenizer(response, return_tensors="pt").input_ids.to(device)

# 3. 챗봇 실행
if __name__ == "__main__":
    chat()

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/kakaobrain/kogpt.
401 Client Error. (Request ID: Root=1-68c40aaa-6addafe82eccad2d3714141b;2a0c104f-a151-4a4d-9092-eef20c4271a4)

Cannot access gated repo for url https://huggingface.co/kakaobrain/kogpt/resolve/main/config.json.
Access to model kakaobrain/kogpt is restricted. You must have access to it and be authenticated to access it. Please log in.